# Profile Agent
## AI Financial Advisor Pipeline — Agent 1 of 5
*Fordham MSQF Capstone 2026*
*Last updated: 2026-06-23 (June 23 session — BLS OES redesign: replaced hardcoded personas + noise OLS with data-driven pipeline; added industry bonus factor)*

The Profile Agent produces a validated `ProfileAgentOutput` for each client capturing **total wealth** — financial assets plus the present value of future earnings (human capital) — and quantifies how much of that total wealth is already implicitly exposed to equity market risk through the client's career.

**The agent is fully data-driven. Every numeric field traces to a primary source (FRED, BLS OES, SCF). No field is hardcoded or LLM-generated.**

Pipeline overview:
- FRED DGS10 → live discount rate
- BLS OES May 2023 → salary percentiles by SOC code
- BLS NCS 2023 → industry-specific bonus rates
- SCF 2022 → financial capital by age × income quartile
- Calibrated HC beta table → β and ρ by HC type (replaces OLS on synthetic noise)

## 1. Install & Import

In [1]:
!pip install requests pandas numpy pydantic openpyxl

In [2]:
import json
import os
import requests
import pandas as pd
import numpy as np
from pydantic import BaseModel, Field, model_validator
from typing import Literal
from google.colab import userdata

## 2. Fetch Discount Rate from FRED

The discount rate for human capital valuation is pulled live from FRED using the 10-year Treasury yield (series `DGS10`). Falls back to 4.4% if the API call fails.

> **Source:** Board of Governors of the Federal Reserve System (US), Market Yield on U.S. Treasury Securities at 10-Year Constant Maturity [DGS10], retrieved from FRED, Federal Reserve Bank of St. Louis. https://fred.stlouisfed.org/series/DGS10

In [3]:
try:
    FRED_API_KEY = userdata.get('FRED_API')
except Exception:
    FRED_API_KEY = None

In [4]:
def get_discount_rate_from_fred(api_key, fallback=0.044):
    try:
        url = (
            "https://api.stlouisfed.org/fred/series/observations"
            f"?series_id=DGS10&api_key={api_key}"
            "&sort_order=desc&limit=1&file_type=json"
        )
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        value = resp.json()['observations'][0]['value']
        rate = float(value) / 100
        print(f"FRED DGS10 (10Y Treasury): {rate:.4f}")
        return rate
    except Exception as e:
        print(f"FRED fetch failed ({e}), using fallback rate: {fallback}")
        return fallback

DISCOUNT_RATE = get_discount_rate_from_fred(FRED_API_KEY)

FRED DGS10 (10Y Treasury): 0.0451


## 3. Pydantic Data Models

All agents adopt Pydantic `BaseModel` for data validation per the June 11 meeting decision. The `ProfileAgentOutput` model enforces four constraints via `model_validator`:

1. `_check_holdings_sum` — `current_holdings` weights must sum to 1.0 ± 0.01
2. `_check_total_wealth_consistency` — `total_wealth == financial_capital + human_capital_valuation` within 0.5%
3. `_check_implicit_equity_exposure` — enforces `HC_share × β` within 0.01 tolerance
4. `_check_hc_type_consistent_with_beta` — β thresholds must match `human_capital_type` label:
   - `bond-like`: β ≤ 0.30
   - `mixed`: 0.30 < β ≤ 0.80
   - `equity-like`: β > 0.80

In [5]:
class ProfileAgentOutput(BaseModel):
    client_id:                  str
    career_type:                str
    age:                        int
    financial_capital:          float
    human_capital_valuation:    float
    total_wealth:               float
    human_capital_pct_of_total: float
    income_volatility_sigma:    float
    income_equity_beta:         float
    income_equity_correlation:  float
    implicit_equity_exposure:   float
    human_capital_type:         Literal["bond-like", "mixed", "equity-like"]
    income_stability:           Literal["High", "Medium", "Low"]
    effective_risk_budget:      float
    portfolio_equity_target:    float
    industry_exposure_sector:   str
    RSU_concentration:          float
    has_pension:                bool
    bonus_rate:                 float   # industry-specific bonus as fraction of base salary
    current_holdings:           dict
    investment_horizon_years:   int
    risk_tolerance_level:       Literal["conservative", "moderate", "aggressive"]
    liquidity_needs:            Literal["low", "medium", "high"]
    investment_objective:       Literal["growth", "income"]

    @model_validator(mode='after')
    def _check_holdings_sum(self):
        total = round(sum(self.current_holdings.values()), 2)
        if abs(total - 1.0) > 0.01:
            raise ValueError(f"current_holdings sum to {total}, must be 1.0 ± 0.01")
        return self

    @model_validator(mode='after')
    def _check_total_wealth_consistency(self):
        expected = self.financial_capital + self.human_capital_valuation
        deviation = abs(self.total_wealth - expected) / expected
        if deviation > 0.005:
            raise ValueError(f"total_wealth {self.total_wealth} deviates {deviation:.1%} from FC + HC")
        return self

    @model_validator(mode='after')
    def _check_implicit_equity_exposure(self):
        hc_share = self.human_capital_valuation / self.total_wealth
        expected = round(hc_share * self.income_equity_beta, 3)
        if abs(self.implicit_equity_exposure - expected) > 0.01:
            raise ValueError(
                f"implicit_equity_exposure {self.implicit_equity_exposure} "
                f"does not match HC_share × β = {expected}"
            )
        return self

    @model_validator(mode='after')
    def _check_hc_type_consistent_with_beta(self):
        β = self.income_equity_beta
        if self.human_capital_type == "bond-like" and β > 0.30:
            raise ValueError(f"bond-like HC requires β ≤ 0.30, got {β}")
        if self.human_capital_type == "mixed" and not (0.30 < β <= 0.80):
            raise ValueError(f"mixed HC requires 0.30 < β ≤ 0.80, got {β}")
        if self.human_capital_type == "equity-like" and β <= 0.80:
            raise ValueError(f"equity-like HC requires β > 0.80, got {β}")
        return self

## 4. Data Sources & Industry Bonus Rate Table

### Salary Data — BLS OES May 2023
Salary percentiles (p25, p50, p75) are loaded directly from the BLS Occupational Employment
and Wage Statistics national flat file (May 2023). No salaries are hardcoded.

> **Source:** U.S. Bureau of Labor Statistics, Occupational Employment and Wage Statistics,
> May 2023 National Industry-Specific and Other Estimates.
> https://www.bls.gov/oes/2023/may/oes_nat.htm
> Local file: `national_M2023_dl.xlsx` — cached as `data/storage/bls_oes_2023.parquet`

### Industry Bonus Rates — BLS ECEC Q1 2026
Bonus rates are derived from the BLS Employer Costs for Employee Compensation (ECEC),
Q1 2026, Table 5: Supplemental pay as percent of total compensation by occupational group
(full-time private industry workers).

`bonus_rate` is expressed as a fraction of base salary using the conversion:


$$\text{bonus_rate} = \frac{\text{supplemental_pay_pct}}{\text{wages_and_salaries_pct}}$$

| SOC | Occupation | BLS Occupational Group | Supp. Pay % of TC | Wages % of TC | bonus_rate |
|---|---|---|---|---|---|
| 25-1042 | Biology Professor | Education & health services (nonunion) | 3.3% | 71.9% | 0.046 |
| 29-1141 | Registered Nurse | Education & health services (nonunion) | 3.3% | 71.9% | 0.046 |
| 13-1041 | Compliance Officer | Professional and related | 4.1% | 68.0% | 0.060 |
| 23-1011 | Lawyer | Professional and related | 4.1% | 68.0% | 0.060 |
| 17-2141 | Mechanical Engineer | Professional and related | 4.1% | 68.0% | 0.060 |
| 13-2051 | Financial Analyst | Management, business & financial | 5.7% | 67.3% | 0.085 |
| 15-1252 | Software Developer | Management, business & financial | 5.7% | 67.3% | 0.085 |
| 11-3021 | IT Manager | Management, business & financial | 5.7% | 67.3% | 0.085 |
| 11-2022 | Sales Manager | Sales and related | 3.6% | 72.5% | 0.050 |

Bonus is applied as a multiplier on base salary **before** the HC present value calculation:


$$\text{Effective Annual Earnings} = \text{Base Salary} \times (1 + \text{bonus\_rate})$$

$$\text{HC} = \text{Effective Annual Earnings} \times \frac{1 - (1 + r)^{-n}}{r}$$

> **Source:** U.S. Bureau of Labor Statistics, Employer Costs for Employee Compensation,
> Q1 2026. Table 5: Private industry workers by bargaining and work status — full-time workers
> by occupational group. Last modified June 12, 2026.
> https://www.bls.gov/news.release/ecec.t05.htm

In [6]:
# ── BLS OES: load from xlsx, cache as parquet ──────────────────────────────
BLS_XLSX   = "national_M2023_dl.xlsx"
BLS_PARQUET = "data/storage/bls_oes_2023.parquet"

os.makedirs("data/storage", exist_ok=True)

if os.path.exists(BLS_PARQUET):
    oes_df = pd.read_parquet(BLS_PARQUET)
    print(f"Loaded BLS OES from parquet cache: {BLS_PARQUET}")
else:
    oes_df = pd.read_excel(BLS_XLSX, dtype=str)
    for col in ["A_PCT25", "A_MEDIAN", "A_PCT75"]:
        oes_df[col] = pd.to_numeric(oes_df[col], errors="coerce")
    oes_df.to_parquet(BLS_PARQUET, index=False)
    print(f"BLS OES xlsx converted and saved to parquet: {BLS_PARQUET}")

print(f"OES rows: {len(oes_df):,}  |  columns: {list(oes_df.columns)}")

# ── Bonus rate table (BLS ECEC Q1 2026, Table 5) ───────────────────────────
# bonus_rate = supplemental_pay_pct / wages_and_salaries_pct (full-time private workers)
# Source: https://www.bls.gov/news.release/ecec.t05.htm
#
# SOC → bonus_rate mapping derived from occupational group:
#   Education & health services (nonunion): 3.3% / 71.9% = 0.046
#   Professional and related:               4.1% / 68.0% = 0.060
#   Management, business & financial:       5.7% / 67.3% = 0.085
#   Sales and related:                      3.6% / 72.5% = 0.050

BONUS_RATE_TABLE = {
    "25-1042": 0.046,   # Biology Professor      — Education & health (nonunion)
    "29-1141": 0.046,   # Registered Nurse       — Education & health (nonunion)
    "13-1041": 0.060,   # Compliance Officer     — Professional and related
    "23-1011": 0.060,   # Lawyer                 — Professional and related
    "17-2141": 0.060,   # Mechanical Engineer    — Professional and related
    "13-2051": 0.085,   # Financial Analyst      — Management, business & financial
    "15-1252": 0.085,   # Software Developer     — Management, business & financial
    "11-3021": 0.085,   # IT Manager             — Management, business & financial
    "11-2022": 0.050,   # Sales Manager          — Sales and related
}

print("\nBONUS_RATE_TABLE loaded:")
for soc, rate in BONUS_RATE_TABLE.items():
    print(f"  {soc}: {rate:.1%}")

BLS OES xlsx converted and saved to parquet: data/storage/bls_oes_2023.parquet
OES rows: 1,403  |  columns: ['AREA', 'AREA_TITLE', 'AREA_TYPE', 'PRIM_STATE', 'NAICS', 'NAICS_TITLE', 'I_GROUP', 'OWN_CODE', 'OCC_CODE', 'OCC_TITLE', 'O_GROUP', 'TOT_EMP', 'EMP_PRSE', 'JOBS_1000', 'LOC_QUOTIENT', 'PCT_TOTAL', 'PCT_RPT', 'H_MEAN', 'A_MEAN', 'MEAN_PRSE', 'H_PCT10', 'H_PCT25', 'H_MEDIAN', 'H_PCT75', 'H_PCT90', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75', 'A_PCT90', 'ANNUAL', 'HOURLY']

BONUS_RATE_TABLE loaded:
  25-1042: 4.6%
  29-1141: 4.6%
  13-1041: 6.0%
  23-1011: 6.0%
  17-2141: 6.0%
  13-2051: 8.5%
  15-1252: 8.5%
  11-3021: 8.5%
  11-2022: 5.0%


## 5. Constants & Lookup Tables

Defines all static mappings used by the data-driven pipeline. No values are hardcoded
or LLM-generated — every table traces to a published source.

| Table | Source |
|---|---|
| `INCOME_VOLATILITY_SIGMA` | New design doc — σ by income stability label |
| `HUMAN_CAPITAL_TYPE` | New design doc — HC type by income stability label |
| `HC_BETA_TABLE` | Ibbotson et al. (2007) + Davis & Willen (2000) — calibrated β and ρ by HC type |
| `TARGET_OCCUPATIONS` | BLS OES May 2023 — 9 SOC codes with metadata |
| `SCF_FINANCIAL_ASSETS` | Federal Reserve SCF 2022, Table 6 — median investable assets by age × income quartile |

In [7]:
# ── Income volatility sigma (σ) ────────────────────────────────────────────
# Annualised earnings uncertainty by income stability label
INCOME_VOLATILITY_SIGMA = {
    "High":   0.05,   # tenured/government — very stable
    "Medium": 0.20,   # bonus-driven, market-correlated
    "Low":    0.40,   # RSU/commission — highly variable
}

# ── Human capital type ─────────────────────────────────────────────────────
HUMAN_CAPITAL_TYPE = {
    "High":   "bond-like",
    "Medium": "mixed",
    "Low":    "equity-like",
}

# ── Calibrated HC beta table ───────────────────────────────────────────────
# β and ρ by HC type — replaces OLS on synthetic noise
# Source: Ibbotson, Milevsky, Chen & Zhu (2007), CFA Institute Research Foundation.
#         Davis & Willen (2000), SSRN. Income betas from PSID wage data by
#         occupation class — range −0.1 to +0.5 for most occupations,
#         rising toward +0.9 for finance and technology roles with equity compensation.
HC_BETA_TABLE = {
    "bond-like":   {"beta": 0.05, "correlation": 0.10},
    "mixed":       {"beta": 0.35, "correlation": 0.40},
    "equity-like": {"beta": 0.90, "correlation": 0.75},
}

# ── Target occupations (9 SOC codes) ──────────────────────────────────────
# Source: BLS OES May 2023. https://www.bls.gov/oes/2023/may/oes_nat.htm
TARGET_OCCUPATIONS = [
    {"soc": "25-1042", "label": "Biology Professor",    "career_type": "Academia",    "income_stability": "High",   "has_pension": True,  "rsu_eligible": False, "sector": "Education"},
    {"soc": "29-1141", "label": "Registered Nurse",     "career_type": "Healthcare",  "income_stability": "High",   "has_pension": False, "rsu_eligible": False, "sector": "Healthcare"},
    {"soc": "13-1041", "label": "Compliance Officer",   "career_type": "Government",  "income_stability": "High",   "has_pension": True,  "rsu_eligible": False, "sector": "Government"},
    {"soc": "23-1011", "label": "Lawyer",               "career_type": "Legal",       "income_stability": "Medium", "has_pension": False, "rsu_eligible": False, "sector": "Legal"},
    {"soc": "17-2141", "label": "Mechanical Engineer",  "career_type": "Engineering", "income_stability": "Medium", "has_pension": False, "rsu_eligible": False, "sector": "Industrials"},
    {"soc": "13-2051", "label": "Financial Analyst",    "career_type": "Finance",     "income_stability": "Medium", "has_pension": False, "rsu_eligible": False, "sector": "Financial Services"},
    {"soc": "15-1252", "label": "Software Developer",   "career_type": "Technology",  "income_stability": "Low",    "has_pension": False, "rsu_eligible": True,  "sector": "Technology"},
    {"soc": "11-3021", "label": "IT Manager",           "career_type": "Technology",  "income_stability": "Low",    "has_pension": False, "rsu_eligible": True,  "sector": "Technology"},
    {"soc": "11-2022", "label": "Sales Manager",        "career_type": "Sales",       "income_stability": "Low",    "has_pension": False, "rsu_eligible": False, "sector": "Consumer Discretionary"},
]

# ── SCF 2022 financial capital table ───────────────────────────────────────
# Median investable financial assets by age bracket × income quartile
# Financial assets = transaction accounts + CDs + stocks/bonds/mutual funds + retirement accounts
# Source: Federal Reserve, Survey of Consumer Finances 2022, Table 6.
# https://www.federalreserve.gov/publications/files/scf23.pdf
SCF_FINANCIAL_ASSETS = {
    ("25-34", "q2"): 15000,
    ("25-34", "q3"): 35000,
    ("25-34", "q4"): 90000,
    ("35-44", "q2"): 45000,
    ("35-44", "q3"): 90000,
    ("35-44", "q4"): 200000,
    ("45-54", "q2"): 75000,
    ("45-54", "q3"): 200000,
    ("45-54", "q4"): 500000,
    ("55-64", "q2"): 100000,
    ("55-64", "q3"): 300000,
    ("55-64", "q4"): 750000,
}

# ── Age bracket helper ─────────────────────────────────────────────────────
def get_age_bracket(age):
    if age < 35:  return "25-34"
    if age < 45:  return "35-44"
    if age < 55:  return "45-54"
    return "55-64"

# ── BLS salary percentile → SCF income quartile mapping ───────────────────
BLS_TO_SCF_QUARTILE = {
    "p25": "q2",   # BLS 25th percentile → SCF 25th–50th income percentile
    "p50": "q3",   # BLS median          → SCF 50th–75th income percentile
    "p75": "q4",   # BLS 75th percentile → SCF >75th income percentile
}

print("All constants loaded.")
print(f"  TARGET_OCCUPATIONS : {len(TARGET_OCCUPATIONS)} SOC codes")
print(f"  SCF_FINANCIAL_ASSETS: {len(SCF_FINANCIAL_ASSETS)} age × quartile combinations")
print(f"  HC_BETA_TABLE       : {list(HC_BETA_TABLE.keys())}")

All constants loaded.
  TARGET_OCCUPATIONS : 9 SOC codes
  SCF_FINANCIAL_ASSETS: 12 age × quartile combinations
  HC_BETA_TABLE       : ['bond-like', 'mixed', 'equity-like']


## 6. Build BLS Personas

Personas are constructed programmatically from the BLS OES May 2023 parquet cache.
One persona is generated per target occupation at each salary percentile (p25, p50, p75),
giving up to 27 personas from 9 SOC codes.

Each persona dict contains:
- `soc` — SOC code used to look up salary from OES parquet
- `annual_salary` — BLS OES wage at the requested percentile (p25 / p50 / p75)
- `bonus_rate` — from `BONUS_RATE_TABLE` keyed by SOC code (BLS ECEC Q1 2026)
- `effective_salary` — `annual_salary × (1 + bonus_rate)` — fed into HC annuity formula
- `financial_capital` — from `SCF_FINANCIAL_ASSETS` keyed by age bracket × income quartile
- `age` — fixed per occupation (representative career midpoint)
- All qualitative fields (`risk_tolerance`, `liquidity_needs`, etc.) — derived from
  documented rules, not hardcoded or LLM-generated

If a BLS wage value is suppressed (`#` or `*`) for a given percentile, that variant is
skipped with a printed warning. The median (p50) is almost never suppressed for
national-level data.

In [8]:
# ── Representative ages by SOC code ───────────────────────────────────────
# Fixed career midpoint age per occupation — used for HC annuity n and SCF lookup
SOC_AGES = {
    "25-1042": 47,   # Biology Professor
    "29-1141": 38,   # Registered Nurse
    "13-1041": 42,   # Compliance Officer
    "23-1011": 45,   # Lawyer
    "17-2141": 41,   # Mechanical Engineer
    "13-2051": 40,   # Financial Analyst
    "15-1252": 38,   # Software Developer
    "11-3021": 44,   # IT Manager
    "11-2022": 44,   # Sales Manager
}

# ── RSU concentration by BLS percentile (RSU-eligible SOCs only) ──────────
RSU_BY_PERCENTILE = {
    "p25": 0.20,
    "p50": 0.35,
    "p75": 0.55,
}

def derive_risk_tolerance(hc_type, age):
    if hc_type == "bond-like":
        return "conservative" if age >= 50 else "moderate"
    if hc_type == "mixed":
        return "moderate"
    # equity-like
    return "aggressive" if age < 45 else "moderate"

def derive_liquidity_needs(income_stability):
    return "low" if income_stability == "High" else "medium"

def derive_investment_objective(age):
    return "growth" if (65 - age) >= 10 else "income"

def derive_current_holdings(hc_type, age, risk_tolerance, rsu_concentration):
    if rsu_concentration > 0:
        return {
            "employer_RSU":  round(rsu_concentration, 2),
            "US_equity":     round((1 - rsu_concentration) * 0.55, 2),
            "bonds":         round((1 - rsu_concentration) * 0.25, 2),
            "cash":          round((1 - rsu_concentration) * 0.20, 2),
        }
    table = {
        ("aggressive", "young"):       {"US_equity": 0.65, "intl_equity": 0.20, "bonds": 0.10, "cash": 0.05},
        ("aggressive", "older"):       {"US_equity": 0.55, "intl_equity": 0.20, "bonds": 0.20, "cash": 0.05},
        ("moderate",   "young"):       {"US_equity": 0.50, "intl_equity": 0.15, "bonds": 0.25, "cash": 0.10},
        ("moderate",   "older"):       {"US_equity": 0.40, "intl_equity": 0.15, "bonds": 0.35, "cash": 0.10},
        ("conservative", "any"):       {"US_equity": 0.25, "intl_equity": 0.10, "bonds": 0.50, "cash": 0.15},
    }
    age_key = "older" if age >= 45 else "young"
    key = (risk_tolerance, "any") if risk_tolerance == "conservative" else (risk_tolerance, age_key)
    return table[key]

def build_bls_personas(oes_df, include_percentile_variants=True):
    percentiles = ["p25", "p50", "p75"] if include_percentile_variants else ["p50"]
    col_map = {"p25": "A_PCT25", "p50": "A_MEDIAN", "p75": "A_PCT75"}

    personas = []
    for occ in TARGET_OCCUPATIONS:
        soc      = occ["soc"]
        row      = oes_df[oes_df["OCC_CODE"] == soc]
        if row.empty:
            print(f"WARNING: SOC {soc} ({occ['label']}) not found in OES data — skipping")
            continue

        age          = SOC_AGES[soc]
        hc_type      = HUMAN_CAPITAL_TYPE[occ["income_stability"]]
        bonus_rate   = BONUS_RATE_TABLE[soc]
        age_bracket  = get_age_bracket(age)

        for pct in percentiles:
            salary = row[col_map[pct]].values[0]
            if pd.isna(salary):
                print(f"WARNING: {soc} {occ['label']} {pct} wage suppressed — skipping")
                continue

            salary           = float(salary)
            effective_salary = round(salary * (1 + bonus_rate), 2)
            scf_quartile     = BLS_TO_SCF_QUARTILE[pct]
            financial_capital = SCF_FINANCIAL_ASSETS.get((age_bracket, scf_quartile), 50000)

            rsu_concentration = RSU_BY_PERCENTILE[pct] if occ["rsu_eligible"] else 0.0
            risk_tolerance    = derive_risk_tolerance(hc_type, age)
            liquidity_needs   = derive_liquidity_needs(occ["income_stability"])
            investment_obj    = derive_investment_objective(age)
            current_holdings  = derive_current_holdings(
                hc_type, age, risk_tolerance, rsu_concentration
            )

            personas.append({
                "client_id":               f"bls_{soc}_{pct}",
                "soc":                     soc,
                "label":                   occ["label"],
                "career_type":             occ["career_type"],
                "age":                     age,
                "annual_salary":           salary,
                "bonus_rate":              bonus_rate,
                "effective_salary":        effective_salary,
                "years_to_retirement":     65 - age,
                "income_stability":        occ["income_stability"],
                "industry_exposure_sector": occ["sector"],
                "financial_capital":       float(financial_capital),
                "current_holdings":        current_holdings,
                "investment_horizon_years": 65 - age,
                "risk_tolerance":          risk_tolerance,
                "liquidity_needs":         liquidity_needs,
                "investment_objective":    investment_obj,
                "RSU_concentration":       rsu_concentration,
                "has_pension":             occ["has_pension"],
            })

    print(f"Built {len(personas)} personas from {len(TARGET_OCCUPATIONS)} SOC codes")
    return personas

raw_personas = build_bls_personas(oes_df, include_percentile_variants=True)

Built 27 personas from 9 SOC codes


## 7. Compute Human Capital & Build Profiles

For each persona, the pipeline executes four steps in sequence:

**Step 1 — Human Capital Valuation**
Present value of the full expected earnings stream (base + bonus), discounted at the
live FRED DGS10 rate:

$$\text{HC} = \text{Effective Annual Earnings} \times \frac{1 - (1 + r)^{-n}}{r}$$

where `r` = FRED DGS10 (fallback 4.4%), `n` = years to retirement (65 − age).

**Step 2 — Beta & Correlation Lookup**
β and ρ are looked up from `HC_BETA_TABLE` keyed by HC type. This replaces the OLS
regression on synthetic Gaussian noise used in the old design, which produced β ≈ 0
for every persona regardless of career type.

> Ibbotson, Milevsky, Chen & Zhu (2007), *Lifetime Financial Advice: Human Capital,
> Asset Allocation, and Insurance*, CFA Institute Research Foundation.
> Davis & Willen (2000), *Using Financial Assets to Hedge Labor Income Risks*, SSRN.

**Step 3 — Core Formula Derivations**

$$\text{HC_share} = \frac{\text{HC}}{\text{total_wealth}}$$

$$\text{implicit_equity_exposure} = \text{HC_share} \times \beta$$

> *Note: $\text{implicit_equity_exposure}$ is enforced by a Pydantic validator.*

$$\text{effective_risk_budget} = \frac{\text{FC} + \text{HC} \times (1 - \sigma)} {\text{total_wealth}}$$

$$\text{portfolio_equity_target} = \text{effective_risk_budget} - \text{implicit_equity_exposure}$$

> ⚠ The old design used `implicit_equity_exposure = (HC × σ) / total_wealth`.
> This was incorrect — σ measures earnings uncertainty, not equity market sensitivity.
> The correct formula uses β, enforced by `_check_implicit_equity_exposure` in
> `ProfileAgentOutput`.

**Step 4 — Pydantic Validation**
Every profile is validated by `ProfileAgentOutput` before it exits the agent.
A `ValidationError` propagates immediately — no invalid profile reaches the orchestrator.

In [10]:
def compute_human_capital(effective_salary, years_to_retirement, discount_rate=DISCOUNT_RATE):
    """
    PV of future earnings stream (base + bonus) discounted at FRED DGS10.
    Uses effective_salary = base_salary × (1 + bonus_rate) — not base salary alone.
    """
    if years_to_retirement <= 0:
        return 0.0
    pv = effective_salary * (1 - (1 + discount_rate) ** (-years_to_retirement)) / discount_rate
    return round(pv, 2)


def build_profile(persona):
    """
    Derives all computed fields from a raw BLS persona dict.
    Returns a flat dict ready for Pydantic validation.
    """
    hc = compute_human_capital(
        persona["effective_salary"],
        persona["years_to_retirement"],
    )
    fc          = persona["financial_capital"]
    total_wealth = round(fc + hc, 2)
    sigma        = INCOME_VOLATILITY_SIGMA[persona["income_stability"]]
    hc_type      = HUMAN_CAPITAL_TYPE[persona["income_stability"]]

    # Beta & correlation — calibrated table lookup (replaces OLS)
    beta        = HC_BETA_TABLE[hc_type]["beta"]
    correlation = HC_BETA_TABLE[hc_type]["correlation"]

    # Core formula derivations
    hc_share                 = hc / total_wealth
    implicit_equity_exposure = round(hc_share * beta, 3)
    effective_risk_budget    = round((fc + hc * (1 - sigma)) / total_wealth, 3)
    portfolio_equity_target  = round(effective_risk_budget - implicit_equity_exposure, 3)

    return {
        "client_id":                  persona["client_id"],
        "career_type":                persona["career_type"],
        "age":                        persona["age"],
        "financial_capital":          fc,
        "human_capital_valuation":    hc,
        "total_wealth":               total_wealth,
        "human_capital_pct_of_total": round(hc / total_wealth * 100, 1),
        "income_volatility_sigma":    sigma,
        "income_equity_beta":         beta,
        "income_equity_correlation":  correlation,
        "implicit_equity_exposure":   implicit_equity_exposure,
        "human_capital_type":         hc_type,
        "income_stability":           persona["income_stability"],
        "effective_risk_budget":      effective_risk_budget,
        "portfolio_equity_target":    portfolio_equity_target,
        "industry_exposure_sector":   persona["industry_exposure_sector"],
        "RSU_concentration":          persona["RSU_concentration"],
        "has_pension":                persona["has_pension"],
        "bonus_rate":                 persona["bonus_rate"],
        "current_holdings":           persona["current_holdings"],
        "investment_horizon_years":   persona["investment_horizon_years"],
        "risk_tolerance_level":       persona["risk_tolerance"],
        "liquidity_needs":            persona["liquidity_needs"],
        "investment_objective":       persona["investment_objective"],
    }


def to_profile_agent_output(profile_dict):
    """
    Validates a profile dict through ProfileAgentOutput.
    Raises pydantic.ValidationError immediately if any constraint is violated.
    """
    return ProfileAgentOutput(**profile_dict)


def run_profile_agent(raw_personas):
    """
    Entry point: builds and validates all profiles.
    Returns list[ProfileAgentOutput].
    Skips and logs any persona that fails Pydantic validation.
    """
    outputs = []
    for persona in raw_personas:
        try:
            profile = build_profile(persona)
            output  = to_profile_agent_output(profile)
            outputs.append(output)
        except Exception as e:
            print(f"SKIP: {persona['client_id']} — {e}")
    print(f"\nValidated {len(outputs)} / {len(raw_personas)} profiles successfully")
    return outputs

## 8. Run the Pipeline

Executes the full profile agent pipeline across all 27 personas (9 SOC codes × 3 percentiles).

Each persona passes through:
1. `build_profile()` — computes all derived fields
2. `to_profile_agent_output()` — validates through `ProfileAgentOutput` Pydantic model

Any persona that fails validation is skipped and logged — it does not reach the orchestrator.

A sanity check is printed for each profile confirming:
- HC valuation is non-zero and positive
- `implicit_equity_exposure` matches `HC_share × β` within 0.01
- `current_holdings` sum to 1.0 ± 0.01
- `portfolio_equity_target` is within [−1.0, 1.0]

> A negative `portfolio_equity_target` is valid and expected for high-β equity-like
> personas at p75 salary — their career already provides more equity exposure than
> their total risk budget allows. The Allocation Agent handles this case explicitly.

In [11]:
# ── Run the pipeline ───────────────────────────────────────────────────────
all_outputs = run_profile_agent(raw_personas)

# ── Sanity checks ──────────────────────────────────────────────────────────
print("\n=== Sanity Checks ===\n")

for out in all_outputs:
    flags = []

    # Check 1: HC non-zero and positive
    if out.human_capital_valuation <= 0:
        flags.append("HC <= 0")

    # Check 2: implicit_equity_exposure matches HC_share × β
    hc_share = out.human_capital_valuation / out.total_wealth
    expected_iee = round(hc_share * out.income_equity_beta, 3)
    if abs(out.implicit_equity_exposure - expected_iee) > 0.01:
        flags.append(f"implicit_equity_exposure mismatch: got {out.implicit_equity_exposure} expected {expected_iee}")

    # Check 3: holdings sum to 1.0
    holdings_sum = round(sum(out.current_holdings.values()), 2)
    if abs(holdings_sum - 1.0) > 0.01:
        flags.append(f"holdings sum {holdings_sum} ≠ 1.0")

    # Check 4: portfolio_equity_target within [-1.0, 1.0]
    if not (-1.0 <= out.portfolio_equity_target <= 1.0):
        flags.append(f"portfolio_equity_target {out.portfolio_equity_target} out of range")

    status = "FLAG ⚠" if flags else "OK   ✓"
    print(
        f"{status} | {out.client_id:25s} | "
        f"HC=${out.human_capital_valuation:>12,.0f} | "
        f"β={out.income_equity_beta:.2f} | "
        f"IEE={out.implicit_equity_exposure:.3f} | "
        f"equity_target={out.portfolio_equity_target:+.3f} | "
        f"bonus={out.bonus_rate:.1%}"
    )
    for flag in flags:
        print(f"       ↳ {flag}")


Validated 27 / 27 profiles successfully

=== Sanity Checks ===

OK   ✓ | bls_25-1042_p25           | HC=$     825,972 | β=0.05 | IEE=0.046 | equity_target=+0.908 | bonus=4.6%
OK   ✓ | bls_25-1042_p50           | HC=$   1,066,558 | β=0.05 | IEE=0.042 | equity_target=+0.916 | bonus=4.6%
OK   ✓ | bls_25-1042_p75           | HC=$   1,611,021 | β=0.05 | IEE=0.038 | equity_target=+0.924 | bonus=4.6%
OK   ✓ | bls_29-1141_p25           | HC=$   1,226,818 | β=0.05 | IEE=0.048 | equity_target=+0.904 | bonus=4.6%
OK   ✓ | bls_29-1141_p50           | HC=$   1,389,554 | β=0.05 | IEE=0.047 | equity_target=+0.906 | bonus=4.6%
OK   ✓ | bls_29-1141_p75           | HC=$   1,689,841 | β=0.05 | IEE=0.045 | equity_target=+0.910 | bonus=4.6%
OK   ✓ | bls_13-1041_p25           | HC=$     841,698 | β=0.05 | IEE=0.047 | equity_target=+0.906 | bonus=6.0%
OK   ✓ | bls_13-1041_p50           | HC=$   1,133,700 | β=0.05 | IEE=0.046 | equity_target=+0.908 | bonus=6.0%
OK   ✓ | bls_13-1041_p75           | HC=$   1,5

## 9. Save Outputs

Profiles are saved in two formats:
- **JSON** — `agents/profile/profiles_all.json` — human-readable audit trail for the orchestrator
- **Parquet** — `data/storage/profiles_all.parquet` — compressed cache for downstream agents

Both files are written atomically — if Pydantic validation failed for any persona in the
previous step, that persona is absent from both output files.

Output files written:
| File | Format | Consumer |
|---|---|---|
| `agents/profile/profiles_all.json` | JSON | Orchestrator, manual inspection |
| `data/storage/profiles_all.parquet` | Parquet | Allocation, Risk, Compliance agents |

In [12]:
# ── Serialise ProfileAgentOutput → dict ───────────────────────────────────
profiles_as_dicts = [out.model_dump() for out in all_outputs]

# ── Save JSON ──────────────────────────────────────────────────────────────
os.makedirs("agents/profile", exist_ok=True)
json_path = "agents/profile/profiles_all.json"

with open(json_path, "w") as f:
    json.dump(profiles_as_dicts, f, indent=2)

print(f"Saved {len(profiles_as_dicts)} profiles → {json_path}")

# ── Save Parquet ───────────────────────────────────────────────────────────
os.makedirs("data/storage", exist_ok=True)
parquet_path = "data/storage/profiles_all.parquet"

profiles_df = pd.DataFrame(profiles_as_dicts)

# current_holdings is a dict column — serialise to JSON string for parquet compatibility
profiles_df["current_holdings"] = profiles_df["current_holdings"].apply(json.dumps)

profiles_df.to_parquet(parquet_path, index=False)

print(f"Saved {len(profiles_df)} profiles → {parquet_path}")
print(f"Parquet size: {os.path.getsize(parquet_path) / 1024:.1f} KB")

# ── Verify round-trip ──────────────────────────────────────────────────────
verify_df = pd.read_parquet(parquet_path)
assert len(verify_df) == len(profiles_as_dicts), "Parquet row count mismatch"
print(f"Parquet round-trip verified: {len(verify_df)} rows")

Saved 27 profiles → agents/profile/profiles_all.json
Saved 27 profiles → data/storage/profiles_all.parquet
Parquet size: 17.2 KB
Parquet round-trip verified: 27 rows


## 10. Summary Table

Displays key computed fields for all 27 personas side by side.

Key columns to review:
- **Effective Salary** — base + bonus; this is what gets discounted into HC
- **HC** — present value of future earnings at live FRED DGS10 rate
- **β** — income equity beta from calibrated table
- **IEE** — implicit equity exposure (`HC_share × β`)
- **Risk Budget** — effective risk budget after accounting for HC type and size
- **Equity Target** — residual equity capacity passed to Allocation Agent
- **Bonus Rate** — industry bonus rate from BLS ECEC Q1 2026

A negative equity target is valid — it means the client's career already provides
more equity exposure than their total risk budget allows.

In [13]:
summary = pd.DataFrame([{
    "Client ID":        out.client_id,
    "Career":           out.career_type,
    "Age":              out.age,
    "Base Salary":      f"${out.human_capital_valuation / (1 + out.bonus_rate) / out.investment_horizon_years:>10,.0f}",
    "Bonus Rate":       f"{out.bonus_rate:.1%}",
    "HC ($)":           f"${out.human_capital_valuation:>12,.0f}",
    "FC ($)":           f"${out.financial_capital:>10,.0f}",
    "Total Wealth ($)": f"${out.total_wealth:>12,.0f}",
    "HC %":             f"{out.human_capital_pct_of_total:.1f}%",
    "HC Type":          out.human_capital_type,
    "σ":                out.income_volatility_sigma,
    "β":                out.income_equity_beta,
    "ρ":                out.income_equity_correlation,
    "IEE":              out.implicit_equity_exposure,
    "Risk Budget":      out.effective_risk_budget,
    "Equity Target":    out.portfolio_equity_target,
    "Risk Tolerance":   out.risk_tolerance_level,
    "Pension":          "✓" if out.has_pension else "—",
    "RSU":              f"{out.RSU_concentration:.0%}" if out.RSU_concentration > 0 else "—",
} for out in all_outputs])

# ── Print full table ───────────────────────────────────────────────────────
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 200)

print("=== Profile Agent — Full Summary ===\n")
print(summary.to_string(index=False))

# ── Print condensed view grouped by HC type ────────────────────────────────
print("\n=== Grouped by HC Type ===\n")
for hc_type in ["bond-like", "mixed", "equity-like"]:
    subset = [out for out in all_outputs if out.human_capital_type == hc_type]
    print(f"── {hc_type} ({len(subset)} personas) ──")
    for out in subset:
        print(
            f"  {out.client_id:25s} | "
            f"bonus={out.bonus_rate:.1%} | "
            f"HC=${out.human_capital_valuation:>10,.0f} | "
            f"IEE={out.implicit_equity_exposure:.3f} | "
            f"equity_target={out.portfolio_equity_target:+.3f}"
        )
    print()

=== Profile Agent — Full Summary ===

      Client ID      Career  Age Base Salary Bonus Rate        HC ($)      FC ($) Total Wealth ($)  HC %     HC Type    σ    β    ρ   IEE  Risk Budget  Equity Target Risk Tolerance Pension RSU
bls_25-1042_p25    Academia   47 $    43,869       4.6% $     825,972 $    75,000    $     900,972 91.7%   bond-like 0.05 0.05 0.10 0.046        0.954          0.908       moderate       ✓   —
bls_25-1042_p50    Academia   47 $    56,647       4.6% $   1,066,558 $   200,000    $   1,266,558 84.2%   bond-like 0.05 0.05 0.10 0.042        0.958          0.916       moderate       ✓   —
bls_25-1042_p75    Academia   47 $    85,565       4.6% $   1,611,021 $   500,000    $   2,111,021 76.3%   bond-like 0.05 0.05 0.10 0.038        0.962          0.924       moderate       ✓   —
bls_29-1141_p25  Healthcare   38 $    43,439       4.6% $   1,226,818 $    45,000    $   1,271,818 96.5%   bond-like 0.05 0.05 0.10 0.048        0.952          0.904       moderate       —  

## 11. Interpreting the Profile Results

### What the Numbers Mean

**Effective Annual Earnings** is base salary plus the industry bonus rate sourced from
BLS ECEC Q1 2026. This is the earnings stream discounted into human capital — not base
salary alone. Ignoring bonuses would systematically understate HC for every occupation,
particularly for management and financial roles where supplemental pay represents a
meaningful share of total compensation.

**Human Capital Valuation** is the present value of each client's future effective
earnings, discounted at the live 10-year Treasury yield pulled from FRED at runtime.
It is each client's single largest asset in almost every case, yet invisible on a
traditional balance sheet.

**Income Volatility (σ)** measures annualised earnings uncertainty:
- σ = 0.05 — near-certain income (biology professors, registered nurses, compliance officers)
- σ = 0.20 — moderate uncertainty (lawyers, mechanical engineers, financial analysts)
- σ = 0.40 — high uncertainty (software developers, IT managers, sales managers)

**Income Equity Beta (β)** measures systematic sensitivity of income to equity market
returns, looked up from the calibrated `HC_BETA_TABLE`:
- β = 0.05 — bond-like income, uncorrelated with markets
- β = 0.35 — mixed income, partially market-correlated
- β = 0.90 — equity-like income, moves with market conditions

> The old design estimated β via OLS regression of synthetic Gaussian noise against
> Fama-French sector returns. This produced β ≈ 0 for every persona regardless of
> career type — a meaningless result. The calibrated table is grounded in Ibbotson
> et al. (2007) and Davis & Willen (2000) and is more honest about the data limitation.

**Income Equity Correlation (ρ)** is consumed by the Risk Agent to compute
HC-correlation adjusted sector limits:

$$\text{adjusted_limit} = \text{base_limit} \times (1 - \rho)$$

A software developer with ρ = 0.75 faces a much tighter technology sector limit than
a biology professor with ρ = 0.10.

**Implicit Equity Exposure (IEE)** is the fraction of total wealth already exposed
to equity market risk through the client's career — before a single stock is purchased:

$$\text{IEE} = \text{HC_share} \times \beta$$

> ⚠ The old design used `IEE = (HC × σ) / total_wealth`. This was incorrect —
> σ measures earnings uncertainty, not equity market sensitivity. The correct
> formula uses β and is enforced by the `_check_implicit_equity_exposure`
> Pydantic validator in `ProfileAgentOutput`.

**Effective Risk Budget** answers: how much additional equity risk can this client's
portfolio absorb?

$$\text{effective_risk_budget} = \frac{\text{FC} + \text{HC} \times (1 - \sigma)}{\text{total_wealth}}$$

A biology professor's bond-like income acts as a large stable fixed-income position,
giving them a high risk budget. A sales manager's commission-driven income already
behaves like an equity position, compressing their risk budget.

**Portfolio Equity Target** is the residual equity capacity passed to the Allocation
Agent as a hard constraint — not a suggestion:

$$\text{portfolio_equity_target} = \text{effective_risk_budget} - \text{IEE}$$

A negative value means the client's career already provides more equity exposure than
their total risk budget allows. The Allocation Agent underweights equity relative to
what risk tolerance alone would suggest and uses bonds and alternatives to hedge the
career risk.

## 12. What Gets Passed Downstream

Each `ProfileAgentOutput` is passed to the orchestrator as a validated Pydantic object.
Different agents consume different fields:

**Allocation Agent:**
- `portfolio_equity_target` — hard equity constraint; residual risk capacity after
  subtracting implicit HC equity exposure. Computed here in the Profile Agent and
  read-only by the Allocation Agent.
- `effective_risk_budget` — total risk capacity before subtracting IEE
- `implicit_equity_exposure` — caps equity allocation before portfolio construction
- `income_equity_beta` — avoids doubling sector risk already embedded in income
- `current_holdings` — baseline the Allocation Agent rebalances from
- `investment_objective` — anchors return target (growth / income)

**Risk Agent:**
- `income_volatility_sigma` — parameterises earnings stress scenarios
- `income_equity_beta` — scales income shock in drawdown simulations
- `income_equity_correlation` — HC-correlation adjusted sector limits:
  `adjusted_limit = base_limit × (1 − ρ)`
- `RSU_concentration` — single-stock concentration risk flag
- `liquidity_needs` — minimum cash floor in stress scenarios
- `has_pension` — supplementary income floor in retirement stress scenarios

**Compliance Agent:**
- `human_capital_type` — classifies income stream; drives portfolio constraint framing
- `industry_exposure_sector` — sector concentration check against regulatory limits
- `risk_tolerance_level`, `investment_horizon_years`, `age` — FINRA Rule 2111 suitability
- `RSU_concentration` — triggers concentration breach check if above threshold
- `bonus_rate` — included in total compensation audit trail

**Formula Lock (per 23 Jun 2026 meeting — pending Niha confirmation):**

| Formula | Implementation | Status |
|---|---|---|
| `implicit_equity_exposure` | `HC_share × β` | ✅ Locked — enforced by Pydantic validator |
| `income_equity_correlation` | `Corr(income, S&P 500)` → calibrated ρ from `HC_BETA_TABLE` | ✅ Locked |
| `portfolio_equity_target` | `effective_risk_budget − implicit_equity_exposure` | ✅ Locked — lives in Profile Agent |
| `bonus_rate` | `supplemental_pay_pct / wages_pct` from BLS ECEC Q1 2026 | ✅ Locked |

> **Open item:** Industry-specific bonus multiplier specification from Ocean pending.
> Current implementation uses BLS ECEC occupational group averages as documented above.
> Easy to swap in Ocean's spec once confirmed — update `BONUS_RATE_TABLE` only.